# Seed-drone scheduling GIS

This notebook keeps only the GIS layers needed for burned-area restoration seed-drone scheduling: burn severity, slope, landslide risk, land cover, a CSV-aligned 1 km x 1 km seed zone, exact 100 m x 100 m subgrids inside it, and seed target pixels for later node generation.


In [1]:
# Kernel smoke test: this cell should finish immediately after a clean restart.
import sys
import time

kernel_started_at = time.perf_counter()
print('kernel ok')
print(sys.executable)


kernel ok
c:\seed_vrp\.venv\Scripts\python.exe


In [2]:
import time

start = time.perf_counter()
import ee
print(f'ee import finished in {time.perf_counter() - start:.2f}s')


ee import finished in 0.33s


In [3]:
PROJECT_ID = 'my-project-495906'

start = time.perf_counter()
try:
    ee.Initialize(project=PROJECT_ID)
    print(f'Earth Engine initialized in {time.perf_counter() - start:.2f}s')
except Exception:
    print('Earth Engine initialization needs authentication.')
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print(f'Earth Engine authenticated and initialized in {time.perf_counter() - start:.2f}s')


Earth Engine initialized in 1.82s


## Analysis Area And Periods


In [4]:
fire_roi = ee.Geometry.Polygon([
    [129.1, 36.8],
    [129.5, 36.8],
    [129.5, 37.3],
    [129.1, 37.3],
    [129.1, 36.8],
])

pre_fire_start = '2022-02-01'
pre_fire_end = '2022-02-28'
post_fire_start = '2022-03-01'
post_fire_end = '2022-04-30'

print('Analysis area and periods are ready.')


Analysis area and periods are ready.


## Burn Severity (dNBR)


In [5]:
S2_SR_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    clear_mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    return image.updateMask(clear_mask).divide(10000)

def add_nbr(image):
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

pre_fire_image = (
    ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(pre_fire_start, pre_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .map(add_nbr)
    .median()
    .clip(fire_roi)
)

post_fire_image = (
    ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(post_fire_start, post_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .map(add_nbr)
    .median()
    .clip(fire_roi)
)

dnbr = pre_fire_image.select('NBR').subtract(
    post_fire_image.select('NBR')
).rename('dNBR')

print('Burn severity layer is ready.')


Burn severity layer is ready.


## Slope And Land Cover


In [6]:
dem = ee.Image('USGS/SRTMGL1_003').clip(fire_roi)
slope = ee.Terrain.slope(dem).rename('Slope_Degrees')

land_cover = (
    ee.Image('ESA/WorldCover/v200/2021')
    .select('Map')
    .clip(fire_roi)
    .rename('Land_Cover')
)

land_cover_class_values = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
land_cover_class_names = {
    10: 'Tree cover',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen',
}
land_cover_for_vis = land_cover.remap(
    land_cover_class_values,
    list(range(1, len(land_cover_class_values) + 1))
).rename('Land_Cover_Class')

print('Slope and land cover layers are ready.')


Slope and land cover layers are ready.


## Landslide Risk Index


In [7]:
dnbr_band = dnbr.select('dNBR')
slope_band = slope.select('Slope_Degrees')

moderate_risk = (
    slope_band.gte(15).And(slope_band.lt(30)).And(dnbr_band.lt(0.4))
).Or(
    slope_band.lt(15).And(dnbr_band.gte(0.2)).And(dnbr_band.lt(0.4))
)
high_risk = slope_band.gte(30).Or(dnbr_band.gte(0.4))
very_high_risk = slope_band.gte(40).And(dnbr_band.gte(0.6))

landslide_risk = (
    ee.Image(1)
    .where(moderate_risk, 2)
    .where(high_risk, 3)
    .where(very_high_risk, 4)
    .rename('Landslide_Risk_Index')
)

burn_mask = dnbr_band.gt(0.2)
landslide_risk_masked = landslide_risk.updateMask(burn_mask)

print('Landslide risk layer is ready.')


Landslide risk layer is ready.


## CSV-Aligned Full 1 km Grid And Seed Zone


In [8]:
import math
import pandas as pd

SEED_PIXEL_DATA_CSV = 'grid_1_10m_subgrid_gis_data.csv'
seed_pixel_df = pd.read_csv(SEED_PIXEL_DATA_CSV)

required_columns = {'latitude', 'longitude'}
missing_columns = required_columns - set(seed_pixel_df.columns)
if missing_columns:
    raise ValueError(f'Missing required CSV columns: {sorted(missing_columns)}')

pixel_count = len(seed_pixel_df)
pixels_per_side = round(math.sqrt(pixel_count))
if pixels_per_side * pixels_per_side != pixel_count:
    raise ValueError(f'Expected a square 10 m pixel grid, got {pixel_count} rows')

csv_lat_min = seed_pixel_df['latitude'].min()
csv_lat_max = seed_pixel_df['latitude'].max()
csv_lon_min = seed_pixel_df['longitude'].min()
csv_lon_max = seed_pixel_df['longitude'].max()

# The CSV stores pixel centers. Expand by half a pixel so the 1 km cell boundary
# fits the 100 x 100 pixel block instead of clipping through center points.
csv_pixel_lat_step = (csv_lat_max - csv_lat_min) / (pixels_per_side - 1)
csv_pixel_lon_step = (csv_lon_max - csv_lon_min) / (pixels_per_side - 1)
CSV_SEED_GRID_BOUNDS = [
    csv_lon_min - csv_pixel_lon_step / 2,
    csv_lat_min - csv_pixel_lat_step / 2,
    csv_lon_max + csv_pixel_lon_step / 2,
    csv_lat_max + csv_pixel_lat_step / 2,
]

GRID_LON_STEP_DEGREES = CSV_SEED_GRID_BOUNDS[2] - CSV_SEED_GRID_BOUNDS[0]
GRID_LAT_STEP_DEGREES = CSV_SEED_GRID_BOUNDS[3] - CSV_SEED_GRID_BOUNDS[1]

fire_coords = fire_roi.getInfo()['coordinates'][0]
roi_west = min(coord[0] for coord in fire_coords)
roi_south = min(coord[1] for coord in fire_coords)
roi_east = max(coord[0] for coord in fire_coords)
roi_north = max(coord[1] for coord in fire_coords)

cols_before_seed_grid = math.ceil((CSV_SEED_GRID_BOUNDS[0] - roi_west) / GRID_LON_STEP_DEGREES - 1e-9)
rows_before_seed_grid = math.ceil((CSV_SEED_GRID_BOUNDS[1] - roi_south) / GRID_LAT_STEP_DEGREES - 1e-9)

grid_origin_lon = CSV_SEED_GRID_BOUNDS[0] - cols_before_seed_grid * GRID_LON_STEP_DEGREES
grid_origin_lat = CSV_SEED_GRID_BOUNDS[1] - rows_before_seed_grid * GRID_LAT_STEP_DEGREES

grid_col_count = math.ceil((roi_east - grid_origin_lon) / GRID_LON_STEP_DEGREES)
grid_row_count = math.ceil((roi_north - grid_origin_lat) / GRID_LAT_STEP_DEGREES)
CSV_SEED_GRID_KEY = f'R{rows_before_seed_grid + 1:02d}_C{cols_before_seed_grid + 1:02d}'

full_1km_grid_specs = []
for row_from_south in range(1, grid_row_count + 1):
    min_lat = grid_origin_lat + (row_from_south - 1) * GRID_LAT_STEP_DEGREES
    max_lat = min_lat + GRID_LAT_STEP_DEGREES
    row_from_north = grid_row_count - row_from_south + 1

    for col_from_west in range(1, grid_col_count + 1):
        min_lon = grid_origin_lon + (col_from_west - 1) * GRID_LON_STEP_DEGREES
        max_lon = min_lon + GRID_LON_STEP_DEGREES
        grid_key = f'R{row_from_south:02d}_C{col_from_west:02d}'

        full_1km_grid_specs.append({
            'grid_key': grid_key,
            'grid_id': len(full_1km_grid_specs) + 1,
            'row_from_south': row_from_south,
            'row_from_north': row_from_north,
            'col_from_west': col_from_west,
            'bounds': [min_lon, min_lat, max_lon, max_lat],
        })

def rectangle_from_bounds(bounds):
    min_lon, min_lat, max_lon, max_lat = bounds
    return ee.Geometry.Polygon([[
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat],
        [min_lon, max_lat],
        [min_lon, min_lat],
    ]])

def feature_from_grid_spec(grid):
    return ee.Feature(
        rectangle_from_bounds(grid['bounds']),
        {
            'grid_key': grid['grid_key'],
            'grid_id': grid['grid_id'],
            'row_from_south': grid['row_from_south'],
            'row_from_north': grid['row_from_north'],
            'col_from_west': grid['col_from_west'],
            'source': 'csv_aligned_full_grid',
        },
    )

full_1km_grid_fc = ee.FeatureCollection([
    feature_from_grid_spec(grid) for grid in full_1km_grid_specs
])

selected_grid_specs = [
    grid for grid in full_1km_grid_specs
    if grid['grid_key'] == CSV_SEED_GRID_KEY
]
if len(selected_grid_specs) != 1:
    raise ValueError(f'Expected 1 CSV-aligned seed grid, got {len(selected_grid_specs)}')

selected_grid_collection = ee.FeatureCollection([
    feature_from_grid_spec(grid) for grid in selected_grid_specs
])
selected_area = selected_grid_collection.geometry()

selected_bounds = selected_grid_specs[0]['bounds']
print(f'CSV pixel rows: {pixel_count}')
print(f'CSV-aligned seed grid key: {CSV_SEED_GRID_KEY}')
print(f'CSV-aligned seed grid bounds: {selected_bounds}')
print(f'Full 1 km grid: {len(full_1km_grid_specs)} cells')


CSV pixel rows: 10000
CSV-aligned seed grid key: R39_C24
CSV-aligned seed grid bounds: [np.float64(129.35299999956925), np.float64(37.13335999677556), np.float64(129.36400000078), np.float64(37.142360133664965)]
Full 1 km grid: 2109 cells


## Exact 100 m x 100 m Subgrids


In [9]:
def make_exact_100m_subgrids(grid):
    grid_key = grid['grid_key']
    grid_id = grid['grid_id']
    min_lon, min_lat, max_lon, max_lat = grid['bounds']
    lon_step = (max_lon - min_lon) / 10
    lat_step = (max_lat - min_lat) / 10

    features = []
    for row in range(10):
        cell_min_lat = min_lat if row == 0 else min_lat + row * lat_step
        cell_max_lat = max_lat if row == 9 else min_lat + (row + 1) * lat_step

        for col in range(10):
            cell_min_lon = min_lon if col == 0 else min_lon + col * lon_step
            cell_max_lon = max_lon if col == 9 else min_lon + (col + 1) * lon_step
            cell_id = f'{grid_key}_r{row + 1:02d}_c{col + 1:02d}'

            geom = ee.Geometry.Polygon([[
                [cell_min_lon, cell_min_lat],
                [cell_max_lon, cell_min_lat],
                [cell_max_lon, cell_max_lat],
                [cell_min_lon, cell_max_lat],
                [cell_min_lon, cell_min_lat],
            ]])
            features.append(ee.Feature(geom, {
                'parent_grid_key': grid_key,
                'parent_grid_id': grid_id,
                'row': row + 1,
                'col': col + 1,
                'cell_id': cell_id,
            }))
    return features

selected_100m_subgrid_features = []
for selected_grid in selected_grid_specs:
    selected_100m_subgrid_features.extend(make_exact_100m_subgrids(selected_grid))

selected_100m_subgrids_fc = ee.FeatureCollection(selected_100m_subgrid_features)

assert len(selected_100m_subgrid_features) == len(selected_grid_specs) * 100
print(f'Exact 100 m subgrids in CSV-aligned seed grid: {len(selected_100m_subgrid_features)}')
print('The CSV-aligned seed grid is split into a 10 x 10 grid with no offset.')


Exact 100 m subgrids in CSV-aligned seed grid: 100
The CSV-aligned seed grid is split into a 10 x 10 grid with no offset.


## Scheduling Inputs And Seed Target Pixels


In [ ]:
# Adjustable targeting rules. These create candidate pixels only; no clustering is done here.
TARGET_DNBR_MIN = 0.44
TARGET_DNBR_MAX = 0.66
TARGET_MAX_SLOPE_DEGREES = 30
SEEDABLE_LAND_COVER_CLASSES = [10, 20, 30, 60]

selected_area_mask = ee.Image.constant(1).clip(selected_area).selfMask()
seedable_land_cover = land_cover.remap(
    SEEDABLE_LAND_COVER_CLASSES,
    [1] * len(SEEDABLE_LAND_COVER_CLASSES),
    0,
).eq(1)

seed_target_mask = (
    dnbr_band.gte(TARGET_DNBR_MIN)
    .And(dnbr_band.lte(TARGET_DNBR_MAX))
    .And(slope_band.lte(TARGET_MAX_SLOPE_DEGREES))
    .And(seedable_land_cover)
    .rename('Seed_Target')
    .updateMask(selected_area_mask)
    .selfMask()
    .clip(selected_area)
)

analysis_stack = ee.Image.cat([
    dnbr,
    slope,
    landslide_risk,
    land_cover,
    seed_target_mask,
]).clip(selected_area)

def add_parent_grid_key(grid):
    def annotate(feature):
        return feature.set({
            'parent_grid_key': grid['grid_key'],
            'parent_grid_id': grid['grid_id'],
            'target_pixel': 1,
        })
    return annotate

seed_target_pixels = ee.FeatureCollection([])
for selected_grid in selected_grid_specs:
    grid_geom = rectangle_from_bounds(selected_grid['bounds'])
    grid_pixels = (
        analysis_stack
        .updateMask(seed_target_mask)
        .sample(
            region=grid_geom,
            scale=10,
            projection=land_cover.projection(),
            geometries=True,
            tileScale=4,
        )
        .map(add_parent_grid_key(selected_grid))
    )
    seed_target_pixels = seed_target_pixels.merge(grid_pixels)

print('Scheduling inputs are ready:')
print('- full_1km_grid_fc')
print('- selected_grid_collection')
print('- selected_100m_subgrids_fc')
print('- analysis_stack')
print('- seed_target_mask')
print('- seed_target_pixels')
print('No clustering is applied yet.')


Scheduling inputs are ready:
- full_1km_grid_fc
- selected_grid_collection
- selected_100m_subgrids_fc
- analysis_stack
- seed_target_mask
- seed_target_pixels
No clustering is applied yet.


## Seed Target Pixel Clustering Into Nodes


In [15]:
# Build scheduling nodes from target pixels using connected components and shape-preserving splits.
# Workload does not need to be equal. Large connected patches are split only to keep nodes spatially meaningful.
import math
import numpy as np
from collections import Counter, deque

CONNECTIVITY_DISTANCE_FACTOR = 1.55  # about 8-neighbor connectivity for a 10 m pixel grid
MAX_PIXELS_PER_NODE = 25             # only large connected patches are split; small patches stay intact

seed_target_info = seed_target_pixels.getInfo()
seed_target_features = seed_target_info.get('features', [])
seed_target_count = len(seed_target_features)

if seed_target_count == 0:
    raise ValueError('No seed target pixels found. Check the target pixel filter conditions first.')

records = []
for index, feature in enumerate(seed_target_features):
    lon, lat = feature['geometry']['coordinates']
    props = feature.get('properties', {})
    records.append({
        'pixel_index': index,
        'longitude': lon,
        'latitude': lat,
        'properties': props,
    })

mean_lat = float(np.mean([record['latitude'] for record in records]))
meters_per_degree_lat = 111_320
meters_per_degree_lon = 111_320 * math.cos(math.radians(mean_lat))
base_lon = records[0]['longitude']
base_lat = records[0]['latitude']

xy = np.array([
    [
        (record['longitude'] - base_lon) * meters_per_degree_lon,
        (record['latitude'] - base_lat) * meters_per_degree_lat,
    ]
    for record in records
], dtype=float)

# Estimate the actual pixel-center spacing, then connect horizontal/vertical/diagonal neighbors.
dx = xy[:, 0][:, None] - xy[:, 0][None, :]
dy = xy[:, 1][:, None] - xy[:, 1][None, :]
distance_sq = dx * dx + dy * dy
np.fill_diagonal(distance_sq, np.inf)
nearest_distance = np.sqrt(np.min(distance_sq, axis=1))
pixel_spacing_m = float(np.median(nearest_distance))
connect_distance_m = pixel_spacing_m * CONNECTIVITY_DISTANCE_FACTOR
connect_distance_sq = connect_distance_m * connect_distance_m

neighbor_cache = [
    np.flatnonzero(distance_sq[i] <= connect_distance_sq).astype(int).tolist()
    for i in range(seed_target_count)
]

# 1) Connected components: pixels that touch as one burned/seedable patch stay together first.
visited = np.zeros(seed_target_count, dtype=bool)
connected_components = []

for start_idx in range(seed_target_count):
    if visited[start_idx]:
        continue

    component = []
    queue = deque([start_idx])
    visited[start_idx] = True

    while queue:
        current = queue.popleft()
        component.append(current)
        for neighbor in neighbor_cache[current]:
            if not visited[neighbor]:
                visited[neighbor] = True
                queue.append(neighbor)

    connected_components.append(component)

# 2) Shape-preserving split: if a connected component is large, peel off compact connected chunks
# from one edge along the component's principal direction. This keeps long/curved patches from being
# represented by one oversized center while avoiding artificial equal-size clusters.
def principal_axis(indices):
    points = xy[indices]
    if len(indices) < 2:
        return np.array([1.0, 0.0])
    centered = points - points.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    return vh[0]

def split_component_preserving_shape(component_indices):
    if len(component_indices) <= MAX_PIXELS_PER_NODE:
        return [component_indices]

    axis = principal_axis(component_indices)
    projection = {idx: float(np.dot(xy[idx], axis)) for idx in component_indices}
    remaining = set(component_indices)
    chunks = []

    while remaining:
        if len(remaining) <= MAX_PIXELS_PER_NODE:
            chunks.append(sorted(remaining, key=lambda idx: projection[idx]))
            break

        # Start from an edge of the remaining patch, then grow through connected target pixels.
        seed_idx = min(remaining, key=lambda idx: projection[idx])
        chunk = []
        frontier = {seed_idx}

        while frontier and len(chunk) < MAX_PIXELS_PER_NODE:
            current = min(frontier, key=lambda idx: np.linalg.norm(xy[idx] - xy[seed_idx]))
            frontier.remove(current)
            if current not in remaining:
                continue

            remaining.remove(current)
            chunk.append(current)

            for neighbor in neighbor_cache[current]:
                if neighbor in remaining:
                    frontier.add(neighbor)

        # A safety fallback for rare cases where prior splits fragment the remaining pixels.
        if not chunk:
            seed_idx = min(remaining, key=lambda idx: projection[idx])
            distances = np.linalg.norm(xy[list(remaining)] - xy[seed_idx], axis=1)
            nearest = [list(remaining)[i] for i in np.argsort(distances)[:MAX_PIXELS_PER_NODE]]
            for idx in nearest:
                remaining.remove(idx)
            chunk = nearest

        chunks.append(sorted(chunk, key=lambda idx: projection[idx]))

    return chunks

node_member_indices = []
for component in connected_components:
    node_member_indices.extend(split_component_preserving_shape(component))

node_features = []
clustered_pixel_features = []

for cluster_id, member_indices in enumerate(node_member_indices, start=1):
    member_records = [records[idx] for idx in member_indices]
    member_xy = xy[member_indices]
    centroid_xy = member_xy.mean(axis=0)

    # Medoid: use an actual target pixel nearest to the cluster centroid as the node location.
    medoid_local_idx = int(np.argmin(np.linalg.norm(member_xy - centroid_xy, axis=1)))
    medoid_idx = member_indices[medoid_local_idx]
    medoid_record = records[medoid_idx]

    dnbr_values = [record['properties'].get('dNBR') for record in member_records if record['properties'].get('dNBR') is not None]
    slope_values = [record['properties'].get('Slope_Degrees') for record in member_records if record['properties'].get('Slope_Degrees') is not None]
    risk_values = [record['properties'].get('Landslide_Risk_Index') for record in member_records if record['properties'].get('Landslide_Risk_Index') is not None]
    land_cover_values = [record['properties'].get('Land_Cover') for record in member_records if record['properties'].get('Land_Cover') is not None]
    parent_grid_keys = [record['properties'].get('parent_grid_key') for record in member_records if record['properties'].get('parent_grid_key')]

    parent_grid_key = Counter(parent_grid_keys).most_common(1)[0][0] if parent_grid_keys else None
    dominant_land_cover = Counter(land_cover_values).most_common(1)[0][0] if land_cover_values else None

    node_id = f'N{cluster_id:03d}'
    node_properties = {
        'node_id': node_id,
        'cluster_id': cluster_id,
        'pixel_count': len(member_indices),
        'parent_grid_key': parent_grid_key,
        'component_count': len(connected_components),
        'representative_pixel_index': int(medoid_record['pixel_index']),
        'mean_dnbr': float(np.mean(dnbr_values)) if dnbr_values else None,
        'mean_slope_degrees': float(np.mean(slope_values)) if slope_values else None,
        'mean_landslide_risk': float(np.mean(risk_values)) if risk_values else None,
        'dominant_land_cover': dominant_land_cover,
    }
    node_features.append(ee.Feature(
        ee.Geometry.Point([medoid_record['longitude'], medoid_record['latitude']]),
        node_properties,
    ))

    for record in member_records:
        pixel_properties = dict(record['properties'])
        pixel_properties.update({
            'node_id': node_id,
            'cluster_id': cluster_id,
            'cluster_pixel_count': len(member_indices),
        })
        clustered_pixel_features.append(
            ee.Feature(ee.Geometry.Point([record['longitude'], record['latitude']]), pixel_properties)
        )

seed_nodes_fc = ee.FeatureCollection(node_features)
clustered_seed_pixels_fc = ee.FeatureCollection(clustered_pixel_features)

node_size_counts = Counter(len(indices) for indices in node_member_indices)
component_size_counts = Counter(len(component) for component in connected_components)
print(f'Seed target pixels: {seed_target_count:,}')
print(f'Connected target patches: {len(connected_components):,}')
print(f'Seed nodes: {len(node_member_indices):,}')
print(f'Estimated pixel spacing: {pixel_spacing_m:.2f} m')
print(f'Connectivity distance: {connect_distance_m:.2f} m')
print(f'Node pixel-count summary: min={min(node_size_counts)}, max={max(node_size_counts)}, distribution={dict(sorted(node_size_counts.items()))}')
print(f'Largest connected patch: {max(component_size_counts)} pixels')
print('Outputs: seed_nodes_fc, clustered_seed_pixels_fc')


Seed target pixels: 4,415
Connected target patches: 76
Seed nodes: 271
Estimated pixel spacing: 7.97 m
Connectivity distance: 12.36 m
Node pixel-count summary: min=1, max=25, distribution={1: 43, 2: 17, 3: 12, 4: 8, 5: 5, 6: 3, 7: 1, 8: 5, 9: 1, 10: 1, 12: 6, 13: 3, 14: 2, 15: 1, 18: 5, 19: 2, 20: 2, 21: 1, 22: 2, 24: 1, 25: 150}
Largest connected patch: 633 pixels
Outputs: seed_nodes_fc, clustered_seed_pixels_fc


## Map Check


In [16]:
import geemap

dnbr_vis_params = {
    'min': -0.5,
    'max': 1.0,
    'palette': ['#006400', '#00FF00', '#FFFF00', '#FFA500', '#FF0000', '#8B0000'],
}
slope_vis_params = {
    'min': 0,
    'max': 45,
    'palette': ['#E0F2FE', '#0284C7', '#082F49'],
}
landslide_risk_vis_params = {
    'min': 1,
    'max': 4,
    'palette': ['#22C55E', '#FACC15', '#F97316', '#DC2626'],
}
land_cover_vis_params = {
    'min': 1,
    'max': 11,
    'palette': [
        '#006400', '#FFBB22', '#FFFF4C', '#F096FF', '#FA0000', '#B4B4B4',
        '#F0F0F0', '#0064C8', '#0096A0', '#00CF75', '#FAE6A0',
    ],
}
seed_target_vis_params = {
    'min': 1,
    'max': 1,
    'palette': ['#FF00FF'],
    'opacity': 0.95,
}

m = geemap.Map(center=[37.03, 129.32], zoom=12, height='800px')

m.add_layer(pre_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Pre-Fire RGB', shown=False)
m.add_layer(post_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Post-Fire RGB', shown=False)
m.add_layer(dnbr, dnbr_vis_params, 'Burn Severity (dNBR)')
m.add_layer(slope, slope_vis_params, 'Slope', shown=False)
m.add_layer(landslide_risk_masked, landslide_risk_vis_params, 'Landslide Risk Index', shown=False)
m.add_layer(land_cover_for_vis, land_cover_vis_params, 'Land Cover', shown=False)

full_1km_grid_style = full_1km_grid_fc.style(
    color='6B7280',
    fillColor='00000000',
    width=0.7,
)
m.add_layer(full_1km_grid_style, {}, 'Full 1km Grid')

selected_100m_grid_style = selected_100m_subgrids_fc.style(
    color='111827',
    fillColor='00000000',
    width=0.9,
)
m.add_layer(selected_100m_grid_style, {}, 'Exact 100m Grids In Selected Cells')

# The seed target pixels are shown once, as one clipped raster layer.
# seed_target_pixels is kept as the future clustering input but is not drawn as a second color layer.
m.add_layer(seed_target_mask, seed_target_vis_params, 'Seed Target Pixels')
m.add_layer(
    seed_nodes_fc.style(color='FFFF00', fillColor='FFFF00CC', pointSize=7, width=1),
    {},
    'Seed Nodes',
)

m.centerObject(selected_grid_collection, zoom=14)
m


Map(center=[37.13786010307806, 129.35850000015756], controls=(WidgetControl(options=['position', 'transparent_…

In [18]:
# Node pixel-count distribution
node_info = seed_nodes_fc.aggregate_array("pixel_count").getInfo()

from collections import Counter
pixel_count_distribution = Counter(node_info)

print(f"Seed nodes: {len(node_info):,}개")
print("Pixels per node distribution:")
for pixel_count, node_count in sorted(pixel_count_distribution.items()):
    print(f"{pixel_count} pixels/node: {node_count:,} nodes")

Seed nodes: 271개
Pixels per node distribution:
1 pixels/node: 43 nodes
2 pixels/node: 17 nodes
3 pixels/node: 12 nodes
4 pixels/node: 8 nodes
5 pixels/node: 5 nodes
6 pixels/node: 3 nodes
7 pixels/node: 1 nodes
8 pixels/node: 5 nodes
9 pixels/node: 1 nodes
10 pixels/node: 1 nodes
12 pixels/node: 6 nodes
13 pixels/node: 3 nodes
14 pixels/node: 2 nodes
15 pixels/node: 1 nodes
18 pixels/node: 5 nodes
19 pixels/node: 2 nodes
20 pixels/node: 2 nodes
21 pixels/node: 1 nodes
22 pixels/node: 2 nodes
24 pixels/node: 1 nodes
25 pixels/node: 150 nodes
